In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
spark=SparkSession.builder.appName("practice").getOrCreate()


In [0]:
from pyspark.sql.types import *
data=[("AAA",25000,4),("BBB",15000,2),("CCC",80000,10)]
schema= StructType([
    StructField("name",StringType()),
    StructField("salary",DoubleType()),
    StructField("experience",IntegerType())
])
df=spark.createDataFrame(data,schema)
display(df)

In [0]:
from pyspark.sql.functions import col, when

df = df.withColumn(
    "Hike",
    when(col("experience") <= 2, col("salary") * 0.10)
    .when((col("experience") > 2) & (col("experience") <= 5), col("salary") * 0.25)
    .otherwise(col("salary") * 0.35)  
)
df.show()


In [0]:
df=df.withColumn("final_salary",col("salary")+col("hike"))
display(df)

In [0]:
df=df.withColumn(
    "joining_date",
    expr("add_months(current_date(),-experience*12)")
              )
display(df)

In [0]:
df = df.withColumn(
    "experience_years",
    floor(months_between(current_date(), col("joining_date")) / 12)
)
display(df)


In [0]:
df = df.withColumn(
    "joining_date",
    date_format(col("joining_date"), "dd-MM-yyyy")
)
display(df)


In [0]:
data = [
    (101, "Laptop", "Electronics", "Asia", 500),
    (102, "Shoes", "Fashion", "Europe", 300),
    (103, "Coffee", "Beverages", "North America", 800),
    (104, "Mobile", "Electronics", "India", 650),
    (105, "Tablet", "Electronics", "South America", 400),
    (106, "Watch", "Fashion", "Middle East", 250),
    (107, "Tea", "Beverages", "Asia", 700),
    (108, "Headphones", "Electronics", "Europe", 350),
    (109, "Jacket", "Fashion", "North America", 450),
    (110, "Juice", "Beverages", "Africa", 600),
    (111, "Camera", "Electronics", "Asia", 275),
    (112, "T-Shirt", "Fashion", "Europe", 500),
    (113, "Soda", "Beverages", "South America", 720),
    (114, "Smartwatch", "Electronics", "India", 320),
    (115, "Bag", "Fashion", "Middle East", 410),
    (116, "Green Tea", "Beverages", "Asia", 560),
    (117, "Speaker", "Electronics", "Europe", 290),
    (118, "Jeans", "Fashion", "North America", 380),
    (119, "Milk", "Beverages", "Africa", 640),
    (120, "Drone", "Electronics", "Asia", 150),
    (121, "Sunglasses", "Fashion", "Europe", 270),
    (122, "Energy Drink", "Beverages", "North America", 900),
    (123, "Desktop", "Electronics", "India", 480),
    (124, "Perfume", "Fashion", "Middle East", 330),
    (125, "Water", "Beverages", "South America", 810),
]
columns = ["product_id", "name", "category", "region","sales"]
df = spark.createDataFrame(data, columns)
df.show(truncate=False)


In [0]:
from pyspark.sql import Window
windowSpec = Window.partitionBy("region").orderBy(col("sales").desc())
df_ranked = df.withColumn("row_num", row_number().over(windowSpec))
df_top5 = df_ranked.filter(col("row_num") <= 5).drop("row_num")
df_top5.show()

In [0]:
data = [
    (1, "Alice", "alice@gmail.com"),
    (2, "Bob", "bob@yahoo.in"),
    (3, "Charlie", "charlie@outlook.com"),
    (4, "David", "david@company.org"),
    (5, "Eva", "eva@hotmail.co.uk"),
    (6, "Frank", "frank@rediffmail.com"),
    (7, "Grace", "grace@icloud.com"),
    (8, "Henry", "henry@protonmail.com"),
    (9, "Ivy", "ivy@zoho.com"),
    (10, "Jack", "jack@live.com"),
]

columns = ["id", "name", "email"]

df = spark.createDataFrame(data, columns)
df.show(truncate=False)

In [0]:
df=df.withColumn(
    "email_domain",
    split(col("email"),"@")[1]
)
display(df)

In [0]:
data=[
(101, "IT", 50000, "Bangalore"),
(102, "IT", 60000, "Bangalore"),
(103, "IT", 45000, "Hyderabad"),
(104, "HR", 35000, "Chennai"),
(105, "HR", 40000, "Chennai"),
(106, "Finance", 55000, "Mumbai"),
(107, "Finance", 65000, "Mumbai"),
(108, "Sales", 30000, "Delhi"),
(109, "Sales", 35000, "Delhi"),
(110, "Sales", 40000, "Delhi")
]

cust_schema=StructType(
    [
        StructField("id",IntegerType()),
        StructField("department",StringType()),
        StructField("salary",IntegerType()),
        StructField("city",StringType())
    ]
)
df=spark.createDataFrame(data,cust_schema)
display(df)

In [0]:
df.groupBy("department").agg(avg("salary").alias("Average"),sum("salary").alias("Total")).show(truncate=False)

In [0]:
schema_transactions = StructType([
    StructField("tx_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("amount", IntegerType(), True),
    StructField("country", StringType(), True),
])

data_transactions = [
    (1001, 2001, 1500, "USA"),
    (1002, 2002, 700, "India"),
    (1003, 2003, 250, "India"),
    (1004, 2004, 50, "USA"),
    (1005, 2005, 1200, "Germany"),
    (1006, 2006, 450, "France"),
    (1007, 2007, 800, "UK"),
    (1008, 2008, 950, "Canada"),
]

transactions = spark.createDataFrame(data_transactions, schema_transactions)

transactions.printSchema()
transactions.show(truncate=False)

In [0]:
transactions=transactions.withColumn(
    "tier",
    when(col("amount")>=1000,"premium")\
        .when((col("amount")>=500) & (col("amount")<1000),"standard")\
        .when((col("amount")>=100) & (col("amount")<500),"basic")\
        .otherwise("low")
    )
display(transactions)

In [0]:
schema_sales = StructType([
    StructField("product_id", StringType(), True),
    StructField("region", StringType(), True),
    StructField("sales", IntegerType(), True),
    StructField("quarter", StringType(), True),
])

data_sales = [
    ("P1", "South", 90000, "Q1"),
    ("P2", "South", 80000, "Q1"),
    ("P3", "South", 75000, "Q1"),
    ("P4", "South", 70000, "Q1"),
    ("P1", "North", 120000, "Q1"),
    ("P5", "North", 100000, "Q1"),
    ("P6", "North", 85000, "Q1"),
    ("P7", "North", 80000, "Q1"),
]

sales_records = spark.createDataFrame(data_sales, schema_sales)

sales_records.printSchema()
sales_records.show(truncate=False)

In [0]:
from pyspark.sql.window import Window
from pyspark.sql.functions import *
windowspec=Window.partitionBy("region").orderBy(col("sales").desc())
reg=sales_records.withColumn("rank",dense_rank().over(windowspec))
reg.filter("rank=1").show()

In [0]:
schema_orders = StructType([
    StructField("order_id", IntegerType(), True),
    StructField("customer_id", IntegerType(), True),
    StructField("order_date", StringType(), True),
    StructField("total", IntegerType(), True),
])

data_orders = [
    (1, 101, "2024-01-01", 100),
    (2, 102, "2024-01-02", 200),
    (3, 103, "2024-01-03", 150),
    (4, 104, "2024-01-04", 300),
    (5, 105, "2024-01-05", 50),
]

orders = spark.createDataFrame(data_orders, schema_orders)

schema_customers = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("customer_name", StringType(), True),
    StructField("city", StringType(), True),
])

data_customers = [
    (101, "Rajesh", "Chennai"),
    (102, "Priya", "Bangalore"),
    (104, "Vikram", "Mumbai"),
    (106, "Anita", "Delhi"),
]

customers = spark.createDataFrame(data_customers, schema_customers)

orders.printSchema()
orders.show(truncate=False)

customers.printSchema()
customers.show(truncate=False)

In [0]:
inner_join=orders.join(customers,orders.customer_id==customers.customer_id,"left")
inner_join.show(truncate=False)

In [0]:
schema_logins = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("login_date", StringType(), True),
    StructField("source", StringType(), True),
    StructField("browser", StringType(), True),
])

data_logins = [
    (1001, "2024-01-01", "web", "Chrome"),
    (1001, "2024-01-01", "web", "Chrome"),
    (1001, "2024-01-01", "mobile", "Firefox"),
    (1002, "2024-01-01", "web", "Safari"),
    (1002, "2024-01-01", "web", "Safari"),
    (1002, "2024-01-02", "mobile", "Chrome"),
]

user_logins = spark.createDataFrame(data_logins, schema_logins)

user_logins.printSchema()
user_logins.show(truncate=False)

In [0]:
df1=user_logins.filter(col("user_id").isNotNull()).dropDuplicates(["login_date","user_id"])
df1.show()

In [0]:
schema_accounts = StructType([
    StructField("account_id", IntegerType(), True),
    StructField("balance", IntegerType(), True),  # nullable
    StructField("account_type", StringType(), True),
    StructField("last_activity", StringType(), True),
])

data_accounts = [
    (1001, None, "savings", "2024-01-01"),
    (1002, 5000, "current", "2024-01-02"),
    (1003, None, "savings", "2024-01-01"),
    (1004, 2000, "current", "2024-01-03"),
    (1005, None, "fixed", "2024-01-01"),
]

accounts = spark.createDataFrame(data_accounts, schema_accounts)

accounts.printSchema()
accounts.show(truncate=False)

In [0]:
accounts.fillna(0,subset=["balance"]).show()

In [0]:
accounts.select(
    coalesce(col("balance"),lit(0))
).show()